# Input File Requirements 

Input file should be an Excel. The table consists of three parts: Symbol column, data columns, and label column.

## 1. Column Structure
 
### **Symbol Column**
- **Column Name**: `Symbol`  
- **Content**: Gene identifiers.  
- **Position**: First column.  
 
### **Data Columns (Dual Time Series)**
- **Column Naming Rule**: `{time}_{dup}_{series}`  
  - `time`: Time point (e.g., 0, 4, 8).  
  - `dup`: Replication number (e.g., 0, 1, 2).  
  - `series`: Time series identifier (0 or 1).  
- **Sorting Rules**:  
  1. **Group by `dup`**: Columns with the same `dup` are grouped together.  
  2. **Sort by `time` within groups**: Within each `dup` group, columns are sorted by `time` in ascending order.  
  3. **Sort groups by `dup`**: Groups are ordered by `dup` (e.g., `dup=0` first, then `dup=1`).  
  4. **All `series=0` columns first, then `series=1` columns**:  
     - First, list **all `series=0` columns** (ordered by `dup` and `time`).  
     - Then, list **all `series=1` columns** (ordered by `dup` and `time`).  
- **Example Column Names**:  
  - Series 1: `0_0_0`, `4_0_0`, `8_0_0`, `0_1_0`, `4_1_0`, `8_1_0`  
  - Series 2: `0_0_1`, `4_0_1`, `8_0_1`, `0_1_1`, `4_1_1`, `8_1_1`  
 
### **Label Columns (4 columns)**
- **Column Names**:  
  - `period_diff`: Indicates **significant difference in period** (1 = significant, 0 = not significant). 
  - `phase_diff`: Indicates **significant difference in phase** (1 = significant, 0 = not significant).  
  - `amp_diff`: Indicates **significant difference in amplitude** (1 = significant, 0 = not significant).  
  - `baseline_diff`: Indicates **significant difference in baseline** (1 = significant, 0 = not significant).  
- **Content**: Binary labels (0 or 1).  
- **Position**: Last four columns.  
- **Note**: If uncertain, set all labels to 1.  
 
---
 
## 2. Example Table Structure
| Symbol | 0_0_0 | 4_0_0 | 8_0_0 | 0_1_0 | 4_1_0 | 8_1_0 | 0_0_1 | 4_0_1 | 8_0_1 | 0_1_1 | 4_1_1 | 8_1_1 | period_diff | phase_diff | amp_diff | baseline_diff |
|--------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------------|------------|----------|---------------|
| GENE1  | 7.342 | 4.928 | 3.213 | 8.029 | 5.750 | 4.837 | 7.401 | 4.950 | 3.250 | 8.102 | 5.780 | 4.850 | 1           | 0          | 1        | 0             |
| GENE2  | 8.029 | 5.750 | 3.421 | 8.258 | 5.849 | 3.729 | 8.102 | 5.780 | 3.450 | 8.301 | 5.880 | 3.750 | 0           | 1          | 0        | 1             |
 
---
 
## 3. Data Filling Instructions
1. **Symbol Column**: Fill in unique gene identifiers.  
2. **Data Columns**:  
   - Populate all `{time}_{dup}_{series}` columns in the correct order.  
3. **Label Columns**:  
   - For each gene, assign 0 or 1 for each difference category.  
   - If uncertain, set all labels to 1. 

In [1]:
from utils.ts import multi_convert_to_ts

# 使用示例 ---------------------------------------------------
# 假设输入文件格式：
# Symbol,0_1,0_2,0_3,3_1,3_2,3_3,...,21_3,label
# gene1,26.39,23.50,22.03,28.25,19.15,22.67,...,23.29,1
# gene2,66.86,63.80,53.70,60.53,60.44,50.98,...,66.91,0

multi_convert_to_ts(
    input_csv='../example_data/example_data_t2.csv',
    output_ts='../example_data/t2/test.ts',
    problem_name='example',
    label_cols=['period_diff', 'phase_diff', 'amp_diff', 'baseline_diff']
)

Multi time-series TS file generated: ../example_data/t2/test.ts


In [2]:
import argparse
from argparse import Namespace
import random
import numpy as np
import os 
import torch
from data_provider.classfication_datasets import MulGroup_MultipleDataset 
from torch.utils.data import DataLoader
from models.circaFM import CIRCAFM  
from peft import PeftModel  
from peft import LoraConfig, get_peft_model
from tqdm import tqdm 
from utils.metrics import Metric  
from tabulate import tabulate  
from datetime import datetime

d:\Software\Anaconda3\envs\circallm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def control_randomness(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = False  # 与训练保持一致
    torch.backends.cudnn.benchmark = True       # 与训练保持一致

In [ ]:
def get_file_paths(folder_path: str) -> list:
    print(f"Current working directory: {os.getcwd()}")
    # If the folder_path itself contains .ts files, return it directly
    ts_files = [f for f in os.listdir(folder_path) if f.endswith('.ts')]
    if ts_files:
        print(f"Found {len(ts_files)} .ts files directly in: {folder_path}")
        return [folder_path]
    # Otherwise, search subdirectories that contain .ts files
    subdirs = [d for d in os.listdir(folder_path)
               if os.path.isdir(os.path.join(folder_path, d))]
    ts_dirs = []
    for sub in subdirs:
        sub_path = os.path.join(folder_path, sub)
        if any(f.endswith('.ts') for f in os.listdir(sub_path)):
            ts_dirs.append(sub_path)
    ts_dirs.sort()
    print(f"Found {len(ts_dirs)} subdirectories with .ts files")
    return ts_dirs


class Circadian_Validator:
    def __init__(self, args: Namespace):
        self.args = args
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._init_model_config()
        self._init_model()
        self._init_dataset()
        self._init_loss_function()

    def _init_model_config(self):
        self.config_dict = {
            "task_name": "diffrhythm",
            "model_name": "CIRCAFM",
            "transformer_type": "encoder_only",
            "freeze_embedder": False,
            "freeze_encoder": False,
            "freeze_head": False,
            "learning_rate": 1e-6,
            "num_epochs": 30,
            "n_channels": 2,
            "n_classes_head1": 2,
            "n_classes_head2": 2,
            "reduction": self.args.reduction,
            "d_model": None,
            "seq_len": self.args.seq_len,
            "enable_gradient_checkpointing": False,
            "enable_FAN": True,
            "enable_FAN_gate": True,
            "patch_len": 6,
            "patch_stride_len": 6,
            "device": self.device.type,
            "transformer_backbone": "google/flan-t5-small",
            "model_kwargs": {},
            "t5_config": {
                "architectures": ["T5ForConditionalGeneration"],
                "d_ff": 1024,
                "d_kv": 64,
                "d_model": 512,
                "decoder_start_token_id": 0,
                "dropout_rate": 0.1,
                "eos_token_id": 1,
                "feed_forward_proj": "gelu",
                "initializer_factor": 1.0,
                "is_encoder_decoder": True,
                "layer_norm_epsilon": 1e-06,
                "model_type": "t5",
                "n_positions": self.args.seq_len,
                "num_decoder_layers": 6,
                "num_heads": 8,
                "num_layers": 6,
                "output_past": True,
                "pad_token_id": 0,
                "relative_attention_max_distance": 128,
                "relative_attention_num_buckets": 32,
                "tie_word_embeddings": False,
                "use_cache": True,
                "vocab_size": 32128
            }
        }
        self.config = Namespace(**self.config_dict)
        print("Model configuration initialized")

    def _verify_loaded_weights(self):
        param_values = []
        for param in self.model.parameters():
            if param.requires_grad:
                param_values.append(param.detach().cpu().numpy().flatten())
        if not param_values:
            print("Warning: Model has no trainable parameters, possible structural error!")
            return

        all_params = np.concatenate(param_values)
        mean_val = np.mean(all_params)
        std_val = np.std(all_params)
        print(f"Trainable param stats: mean={mean_val:.6f}, std={std_val:.6f}")

        if abs(mean_val) < 1e-4:
            for i, param in enumerate(self.model.parameters()):
                if param.requires_grad:
                    print(f"Param {i} first 5 values: {param.detach().cpu().numpy().flatten()[:5]}")
                    if i >= 2:
                        break
        else:
            print("Weights loaded successfully! Trainable parameters far from initial values")

    def _init_model(self):
        self.model = CIRCAFM(self.config)
        self.model.to(self.device)
        print("Base CIRCAFM model initialized")

        if not os.path.isfile(self.args.pretrained_model_path):
            raise FileNotFoundError(f"Model file not found: {self.args.pretrained_model_path}")

        checkpoint = torch.load(
            self.args.pretrained_model_path,
            map_location=self.device,
            weights_only=True
        )
        peft_model_state = checkpoint['model_state_dict']
        print(f"Read PeftModel parameters, total keys: {len(peft_model_state)}")

        sample_key = next(iter(peft_model_state.keys()))
        if "base_model.model." not in sample_key:
            print("Warning: Loaded parameters are not in PeftModel format, may not be a LoRA model")
        else:
            print(f"PeftModel format verified, sample key: {sample_key[:50]}...")

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q", "v"],
            lora_dropout=0.1,
            bias="none",
            task_type="SEQ_CLS"
        )
        self.model = get_peft_model(self.model, lora_config)
        self.model.to(self.device)
        print("Added LoRA structure to base model, converted to PeftModel")

        try:
            load_result = self.model.load_state_dict(peft_model_state, strict=True)
            print("Pretrained weights loaded strictly successfully!")
            if load_result.missing_keys:
                print(f"  Missing keys: {load_result.missing_keys[:3]}...")
            else:
                print("  No missing keys")
            if load_result.unexpected_keys:
                print(f"  Unexpected keys: {load_result.unexpected_keys[:3]}...")
            else:
                print("  No unexpected keys")
        except RuntimeError as e:
            print(f"Strict loading failed, attempting non-strict loading: {str(e)[:300]}")
            load_result = self.model.load_state_dict(peft_model_state, strict=False)
            print(f"  Missing keys count: {len(load_result.missing_keys)}")
            print(f"  Unexpected keys count: {len(load_result.unexpected_keys)}")
            lora_loaded = any("lora_" in key for key in peft_model_state.keys() if key not in load_result.missing_keys)
            if not lora_loaded:
                raise RuntimeError("LoRA adapter parameters not loaded successfully, validation results will be invalid!")

        self._verify_loaded_weights()
        self.model.eval()
        print("Model loaded and set to evaluation mode")

    def _init_dataset(self):
        print(f"Loading validation dataset: {self.args.dataset_path}")
        file_paths = get_file_paths(self.args.dataset_path)
        print("Dataset directories:", file_paths)

        self.val_dataset = MulGroup_MultipleDataset(
            data_split="aper",
            file_paths=file_paths,
            seq_len=self.args.seq_len,
            seed=self.args.seed,
            Realcase=True, 
            normalize_method="together"
        )

        self.val_dataloader = DataLoader(
            self.val_dataset,
            batch_size=self.args.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            drop_last=False
        )
        print(f"Validation dataset loaded: {len(self.val_dataset)} samples, {len(self.val_dataloader)} batches")
        self._analyze_invalid_labels()

    def _analyze_invalid_labels(self):
        labels = self.val_dataset.labels
        total_samples = len(labels)
        total_labels = labels.size

        invalid_mask = labels == -1
        invalid_count = np.sum(invalid_mask)
        invalid_percentage = (invalid_count / total_labels) * 100

        print(f"Invalid label analysis:")
        print(f"  Total samples: {total_samples}")
        print(f"  Total labels: {total_labels}")
        print(f"  Invalid labels: {invalid_count} ({invalid_percentage:.2f}%)")

        for i, category in enumerate(['Period', 'Phase', 'Amplitude', 'Mesor']):
            category_invalid = np.sum(labels[:, i] == -1)
            category_percentage = (category_invalid / total_samples) * 100
            print(f"  {category}: {category_invalid} invalid samples ({category_percentage:.2f}%)")

    def _init_loss_function(self):
        label = self.val_dataset.labels

        def safe_weight(unique_result):
            unique_vals, counts = unique_result
            if len(counts) < 2:
                return 1.0
            return counts[0] / (counts[1] + 1e-8)

        try:
            nn_period = safe_weight(np.unique(label[:, 0][label[:, 0] != -1], return_counts=True))
            nn_phase = safe_weight(np.unique(label[:, 1][label[:, 1] != -1], return_counts=True))
            nn_amp = safe_weight(np.unique(label[:, 2][label[:, 2] != -1], return_counts=True))
            nn_mesor = safe_weight(np.unique(label[:, 3][label[:, 3] != -1], return_counts=True))
        except Exception as e:
            print(f"Warning: Error computing class weights: {e}, using default weight 1.0")
            nn_period = nn_phase = nn_amp = nn_mesor = 1.0

        head1_pos_weights = torch.tensor([nn_period, nn_phase], dtype=torch.float32).to(self.device)
        head2_pos_weights = torch.tensor([nn_amp, nn_mesor], dtype=torch.float32).to(self.device)

        self.criterion_head1 = torch.nn.BCEWithLogitsLoss(pos_weight=head1_pos_weights)
        self.criterion_head2 = torch.nn.BCEWithLogitsLoss(pos_weight=head2_pos_weights)
        self.criterion_head1_2 = torch.nn.BCEWithLogitsLoss(pos_weight=head1_pos_weights, reduction='none')
        self.criterion_head2_2 = torch.nn.BCEWithLogitsLoss(pos_weight=head2_pos_weights, reduction='none')
        print("Loss functions initialized (filtered invalid labels for class weight calculation)")

    def _create_valid_mask(self, targets):
        return targets != -1

    def _mask_invalid_predictions(self, logits, targets, predicted, scores):
        valid_mask = self._create_valid_mask(targets)
        predicted = predicted * valid_mask.int() + (-1) * (~valid_mask).int()
        scores = scores * valid_mask.float() + (-1) * (~valid_mask).float()
        return predicted, scores

    def _calculate_masked_loss(self, logits, targets, criterion):
        valid_mask = self._create_valid_mask(targets)
        loss_per_sample = criterion(logits, targets)
        loss_per_sample = loss_per_sample * valid_mask.float()
        valid_count = valid_mask.sum(dim=0)
        masked_loss = loss_per_sample.sum(dim=0) / (valid_count + 1e-8)
        return masked_loss, valid_count

    def _calculate_masked_accuracy(self, predicted, targets):
        valid_mask = self._create_valid_mask(targets)
        targets_int = targets.int()
        correct_per_sample = (predicted == targets_int) & valid_mask
        correct_count = correct_per_sample.sum(dim=0)
        valid_count = valid_mask.sum(dim=0)
        accuracy_per_class = correct_count.float() / (valid_count.float() + 1e-8)
        return accuracy_per_class, valid_count

    def evaluate(self) -> dict:
        print("Starting evaluation...")
        all_targets, all_preds, all_scores = [], [], []
        sample_metadata = []

        running_loss, t_running_loss, amp_running_loss, phase_running_loss, mesor_running_loss = 0.0, 0.0, 0.0, 0.0, 0.0
        total_valid_T, total_valid_Phase, total_valid_Amp, total_valid_Mesor = 0, 0, 0, 0
        correct_T, correct_Amp, correct_Phase, correct_Mesor = 0, 0, 0, 0

        with torch.no_grad():
            for batch_data_1, input_mask_1, x_marks_1, batch_data_2, input_mask_2, x_marks_2, targets, batch_sample_ids, batch_sample_folders in tqdm(
                self.val_dataloader, total=len(self.val_dataloader), desc="Validating"
            ):
                batch_data_1 = batch_data_1.to(self.device).float()
                batch_data_2 = batch_data_2.to(self.device).float()
                input_mask_1 = input_mask_1.long().to(self.device)
                input_mask_2 = input_mask_2.long().to(self.device)
                x_marks_1 = x_marks_1.to(self.device)
                x_marks_2 = x_marks_2.to(self.device)
                targets = targets.float().to(self.device)

                all_targets.extend(targets.detach().cpu().numpy())

                dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float32
                with torch.autocast(device_type=self.device.type, dtype=dtype, enabled=True):
                    output = self.model(
                        x_enc=batch_data_1,
                        input_mask=input_mask_1,
                        x_mark=x_marks_1,
                        x_enc2=batch_data_2,
                        input_mask2=input_mask_2,
                        x_mark2=x_marks_2,
                        reduction=self.args.reduction,
                        return_dict=True
                    )

                    combined_logits = output.logits
                    metadata = output.metadata
                    head1_classes = metadata['head1_classes']
                    head1_logits = combined_logits[:, :head1_classes]
                    head2_logits = combined_logits[:, head1_classes:]

                    head1_targets = targets[:, :head1_classes]
                    head2_targets = targets[:, head1_classes:]

                    loss_head1, valid_head1 = self._calculate_masked_loss(head1_logits, head1_targets, self.criterion_head1_2)
                    loss_head2, valid_head2 = self._calculate_masked_loss(head2_logits, head2_targets, self.criterion_head2_2)

                    total_loss = (loss_head1.sum() + loss_head2.sum()) / 2

                    period_loss = loss_head1[0] if valid_head1[0] > 0 else torch.tensor(0.0)
                    phase_loss = loss_head1[1] if valid_head1[1] > 0 else torch.tensor(0.0)
                    amp_loss = loss_head2[0] if valid_head2[0] > 0 else torch.tensor(0.0)
                    mesor_loss = loss_head2[1] if valid_head2[1] > 0 else torch.tensor(0.0)

                running_loss += total_loss.item()
                if valid_head1[0] > 0:
                    t_running_loss += period_loss.item()
                if valid_head1[1] > 0:
                    phase_running_loss += phase_loss.item()
                if valid_head2[0] > 0:
                    amp_running_loss += amp_loss.item()
                if valid_head2[1] > 0:
                    mesor_running_loss += mesor_loss.item()

                scores = torch.sigmoid(combined_logits)
                predicted = (scores > 0.5).int()

                predicted, scores = self._mask_invalid_predictions(combined_logits, targets, predicted, scores)

                all_preds.extend(predicted.detach().cpu().numpy())
                all_scores.extend(scores.detach().to(torch.float).cpu().numpy())

                accuracy_per_class, valid_count_per_class = self._calculate_masked_accuracy(predicted, targets)

                correct_T += (accuracy_per_class[0] * valid_count_per_class[0]).int().item()
                correct_Phase += (accuracy_per_class[1] * valid_count_per_class[1]).int().item()
                correct_Amp += (accuracy_per_class[2] * valid_count_per_class[2]).int().item()
                correct_Mesor += (accuracy_per_class[3] * valid_count_per_class[3]).int().item()

                total_valid_T += valid_count_per_class[0].item()
                total_valid_Phase += valid_count_per_class[1].item()
                total_valid_Amp += valid_count_per_class[2].item()
                total_valid_Mesor += valid_count_per_class[3].item()

                if isinstance(batch_sample_ids, (torch.Tensor, np.ndarray)):
                    batch_sample_ids = batch_sample_ids.tolist()
                if isinstance(batch_sample_folders, (torch.Tensor, np.ndarray)):
                    batch_sample_folders = batch_sample_folders.tolist()

                for sid, sfolder in zip(batch_sample_ids, batch_sample_folders):
                    sample_metadata.append({"folder": sfolder, "id": sid})

        num_batches = len(self.val_dataloader)
        avg_loss = running_loss / num_batches if num_batches > 0 else 0.0
        avg_t_loss = t_running_loss / num_batches if num_batches > 0 else 0.0
        avg_phase_loss = phase_running_loss / num_batches if num_batches > 0 else 0.0
        avg_amp_loss = amp_running_loss / num_batches if num_batches > 0 else 0.0
        avg_mesor_loss = mesor_running_loss / num_batches if num_batches > 0 else 0.0

        t_accuracy = correct_T / (total_valid_T + 1e-8) if total_valid_T > 0 else 0.0
        phase_accuracy = correct_Phase / (total_valid_Phase + 1e-8) if total_valid_Phase > 0 else 0.0
        amp_accuracy = correct_Amp / (total_valid_Amp + 1e-8) if total_valid_Amp > 0 else 0.0
        mesor_accuracy = correct_Mesor / (total_valid_Mesor + 1e-8) if total_valid_Mesor > 0 else 0.0

        total_valid_labels = total_valid_T + total_valid_Phase + total_valid_Amp + total_valid_Mesor
        total_correct = correct_T + correct_Phase + correct_Amp + correct_Mesor
        avg_accuracy = total_correct / (total_valid_labels + 1e-8) if total_valid_labels > 0 else 0.0

        loss_result = {
            "avg_loss": avg_loss,
            "avg_t_loss": avg_t_loss,
            "avg_amp_loss": avg_amp_loss,
            "avg_phase_loss": avg_phase_loss,
            "avg_mesor_loss": avg_mesor_loss
        }

        accuracy_result = {
            "avg_accuracy": avg_accuracy,
            "t_accuracy": t_accuracy,
            "amp_accuracy": amp_accuracy,
            "phase_accuracy": phase_accuracy,
            "mesor_accuracy": mesor_accuracy,
            "valid_samples_T": total_valid_T,
            "valid_samples_Phase": total_valid_Phase,
            "valid_samples_Amp": total_valid_Amp,
            "valid_samples_Mesor": total_valid_Mesor
        }

        final_result = {
            "loss": loss_result,
            "accuracy": accuracy_result,
            "sample_metadata": sample_metadata,
            "targets": np.array(all_targets).tolist(),
            "preds": np.array(all_preds).tolist(),
            "scores": np.array(all_scores).tolist(),
            "total_samples": len(all_targets),
            "total_valid_labels": total_valid_labels
        }

        self._print_result(final_result)
        self._save_result(final_result)
        print("Evaluation completed!")
        return final_result

    def _print_result(self, result: dict):
        loss_headers = ["Type", "Avg Loss", "Period Loss", "Phase Loss", "Amp Loss", "Baseline Loss"]
        loss_data = [["Validation",
                      round(result["loss"]["avg_loss"], 4),
                      round(result["loss"]["avg_t_loss"], 4),
                      round(result["loss"]["avg_phase_loss"], 4),
                      round(result["loss"]["avg_amp_loss"], 4),
                      round(result["loss"]["avg_mesor_loss"], 4)]]
        loss_table = tabulate(loss_data, headers=loss_headers, tablefmt="grid", floatfmt=".4f")

        acc_headers = ["Type", "Avg Acc", "Period Acc", "Phase Acc", "Amp Acc", "Baseline Acc", "Valid Samples"]
        acc_data = [["Validation",
                     round(result["accuracy"]["avg_accuracy"], 4),
                     round(result["accuracy"]["t_accuracy"], 4),
                     round(result["accuracy"]["phase_accuracy"], 4),
                     round(result["accuracy"]["amp_accuracy"], 4),
                     round(result["accuracy"]["mesor_accuracy"], 4),
                     f"{result['accuracy']['valid_samples_T']}/{result['accuracy']['valid_samples_Phase']}/{result['accuracy']['valid_samples_Amp']}/{result['accuracy']['valid_samples_Mesor']}"]]
        acc_table = tabulate(acc_data, headers=acc_headers, tablefmt="grid", floatfmt=".4f")

        print("\n" + "=" * 80)
        print(f"Validation Results (Total Samples: {result['total_samples']}, Valid Labels: {result['total_valid_labels']})")
        print("=" * 80)
        print("Loss Results:")
        print(loss_table)
        print("\nAccuracy Results:")
        print(acc_table)
        print("=" * 80 + "\n")

    def _save_result(self, result: dict):
        save_dir = os.path.join(self.args.assets_path, "example")
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "validation_result.json")
        # Assuming Metric.save_metrics exists in the project
        Metric.save_metrics(
            result,
            save_dir,
            "validation_result.json",
            epoch=0,
            c_time='1',
            mode='w'
        )
        print(f"Validation result saved to: {save_path}")

In [5]:
args = Namespace(
    dataset_path="../example_data/t2/",  
    seq_len=72,                        
    reduction="mean",                
    
    pretrained_model_path="pretrained/Task2/best_model.pth",  
    lora=True,                       
    
    batch_size=256,                   
    seed=77,                         
    
    assets_path="../example_data/t2/result/",   
)

# ====================== 执行验证 ======================
if __name__ == "__main__":
    control_randomness(args.seed)
    validator = Circadian_Validator(args)
    validation_result = validator.evaluate()

Model configuration initialized
Base CIRCAFM model initialized
Read PeftModel parameters, total keys: 136
PeftModel format verified, sample key: base_model.model.data_embedding.mask_embedding...
Added LoRA structure to base model, converted to PeftModel
Pretrained weights loaded strictly successfully!
  No missing keys
  No unexpected keys
Trainable param stats: mean=-0.000672, std=0.126638
Weights loaded successfully! Trainable parameters far from initial values
Model loaded and set to evaluation mode
Loading validation dataset: ../example_data/t2/
Current working directory: d:\github\CircaFM\CircaFM_code
Found 1 .ts files directly in: ../example_data/t2/
Dataset directories: ['../example_data/t2/']
Validation dataset loaded: 1000 samples, 4 batches
Invalid label analysis:
  Total samples: 1000
  Total labels: 4000
  Invalid labels: 0 (0.00%)
  Period: 0 invalid samples (0.00%)
  Phase: 0 invalid samples (0.00%)
  Amplitude: 0 invalid samples (0.00%)
  Mesor: 0 invalid samples (0.00%)

Validating: 100%|██████████| 4/4 [00:01<00:00,  3.32it/s]


Validation Results (Total Samples: 1000, Valid Labels: 4000)
Loss Results:
+------------+------------+---------------+--------------+------------+-----------------+
| Type       |   Avg Loss |   Period Loss |   Phase Loss |   Amp Loss |   Baseline Loss |
+============+============+===============+==============+============+=================+
| Validation |     0.6719 |        0.4767 |       0.1972 |     0.3099 |          0.3601 |
+------------+------------+---------------+--------------+------------+-----------------+

Accuracy Results:
+------------+-----------+--------------+-------------+-----------+----------------+---------------------+
| Type       |   Avg Acc |   Period Acc |   Phase Acc |   Amp Acc |   Baseline Acc | Valid Samples       |
+============+===========+==============+=============+===========+================+=====================+
| Validation |    0.9035 |       0.8640 |      0.9500 |    0.9070 |         0.8930 | 1000/1000/1000/1000 |
+------------+-----------+-

In [6]:
import utils.pro_json as pro_json

metrics = pro_json.calculate_multiclass_metrics(json_path='../example_data/t2/result/example/1/validation_result.json')

# 整体（micro‑average）指标
overall = metrics['overall']
print(f"Overall ROC-AUC: {overall['roc_auc']:.4f}")
print(f"Overall AUPR:    {overall['aupr']:.4f}")

# 类别指标
for cls_name, cls_metrics in metrics['class_wise'].items():
    print(f"\n{cls_name}:")
    print(f"  ROC-AUC: {cls_metrics['roc_auc']:.4f}")
    print(f"  AUPR:    {cls_metrics['aupr']:.4f}")


OVERALL METRICS (micro-average)
ROC-AUC: 0.9616
AUPR:    0.9355

Classification Report (overall):
              precision  recall  f1-score   support
0                0.9390  0.9160    0.9273 2689.0000
1                0.8359  0.8780    0.8564 1311.0000
macro avg        0.8874  0.8970    0.8919 4000.0000
weighted avg     0.9052  0.9035    0.9041 4000.0000

CLASS-WISE METRICS

--- period ---
ROC-AUC: 0.9364
AUPR:    0.8729
Classification Report:
              precision  recall  f1-score   support
0                0.9408  0.8647    0.9012  717.0000
1                0.7155  0.8622    0.7821  283.0000
macro avg        0.8282  0.8635    0.8416 1000.0000
weighted avg     0.8771  0.8640    0.8675 1000.0000

--- phase ---
ROC-AUC: 0.9873
AUPR:    0.9661
Classification Report:
              precision  recall  f1-score   support
0                0.9786  0.9514    0.9648  720.0000
1                0.8833  0.9464    0.9138  280.0000
macro avg        0.9310  0.9489    0.9393 1000.0000
weighted avg